# NLP 01: retrieval from metadata (rungs R0 and R1)

Plan: [nlp/EXPERIMENT_PLAN.md](../../src/deep_learning_project_waerniers_lamy/nlp/EXPERIMENT_PLAN.md).
Question: given a visitor's text, which paintings of the collection match best?

Two ways to rank, same data, same metrics:

- **R0** BM25 on the metadata text (classical floor, no neural network).
- **R1** a pretrained sentence encoder (MiniLM) and cosine similarity, no training.

Each painting is described only by its **genre, style and artist**. We search the **validation(A)** corpus (480 paintings);
museum B stays untouched until the final test ([ADR 0003](../../adr/0003-simulated-museum-b-test-set.md)).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer

from deep_learning_project_waerniers_lamy.nlp.utils import load_paintings, ndcg_at_k, precision_at_k, show_ranking

ROOT = Path("..") / ".." / "data" / "wikiart_genre5"
K = 10

## 1. The collection: one short text per painting

In [ ]:
corpus = load_paintings(ROOT, split="val")  # shared split.csv: validation(A), museum B stays untouched

def describe(row: pd.Series) -> str:
    """Metadata as text. 'Unknown Artist' is dropped: it says nothing about the painting."""
    parts = [row.genre.replace("_", " "), row["style"].replace("_", " ")]
    if row.artist != "Unknown Artist":
        parts.append(row.artist.replace("-", " "))
    return ", ".join(parts)

corpus["text"] = corpus.apply(describe, axis=1)
print(len(corpus), "paintings")
corpus[["text"]].head()

## 2. Queries and ground truth

For now only **attribute queries**: the name of a genre or a style. The ground truth comes straight from the labels, so no
human grading is needed. This is a **sanity check, not evidence**: the metadata is both the system's input and the judge
(see plan section 4). The real test, with hand-written content queries, comes later.

In [ ]:
styles = corpus["style"].value_counts()
styles = styles[styles >= 10].index  # keep styles with enough paintings to be meaningful

queries = [(g.replace("_", " "), "genre", g) for g in sorted(corpus.genre.unique())]
queries += [(s.replace("_", " "), "style", s) for s in styles]

def relevance(kind: str, value: str) -> np.ndarray:
    """1 for every painting whose label matches the query, else 0."""
    return (corpus[kind] == value).to_numpy().astype(int)

print(len(queries), "attribute queries, e.g.", [q[0] for q in queries[:8]])

## 3. R0: BM25 (written by hand)

Score of a painting = sum over query words of `idf(word) * tf-saturation`. A word that is rare in the collection counts more;
repeating a word helps less and less. No learning, no vectors of meaning: only exact word matches.

In [ ]:
def tokenize(text: str) -> list[str]:
    return text.lower().replace(",", " ").split()

docs = [tokenize(t) for t in corpus.text]
N = len(docs)
avg_len = np.mean([len(d) for d in docs])
df = {}  # in how many paintings each word appears
for d in docs:
    for w in set(d):
        df[w] = df.get(w, 0) + 1

def bm25_scores(query: str, k1: float = 1.5, b: float = 0.75) -> np.ndarray:
    scores = np.zeros(N)
    for w in tokenize(query):
        if w not in df:
            continue  # a word the collection has never seen carries no signal
        idf = np.log(1 + (N - df[w] + 0.5) / (df[w] + 0.5))
        for i, d in enumerate(docs):
            tf = d.count(w)
            scores[i] += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(d) / avg_len))
    return scores

## 4. R1: pretrained sentence encoder + cosine

`all-MiniLM-L6-v2` (22M parameters) already maps sentences to vectors where similar meanings are close.
We do the pooling ourselves: average the token vectors, then normalise, so that a dot product is a cosine.

In [ ]:
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
encoder = AutoModel.from_pretrained(MODEL).eval()

@torch.no_grad()
def embed(texts: list[str]) -> torch.Tensor:
    """Texts -> unit vectors (n, 384)."""
    batch = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    tokens = encoder(**batch).last_hidden_state              # (n, length, 384)
    mask = batch["attention_mask"].unsqueeze(-1)              # ignore padding tokens
    mean = (tokens * mask).sum(1) / mask.sum(1)               # average over real tokens
    return torch.nn.functional.normalize(mean, dim=-1)

painting_vectors = embed(corpus.text.tolist())                # computed once, offline
print(painting_vectors.shape)

def cosine_scores(query: str) -> np.ndarray:
    return (painting_vectors @ embed([query]).T).squeeze(1).numpy()

## 5. Compare on the attribute queries

In [ ]:
methods = {"R0 BM25": bm25_scores, "R1 MiniLM": cosine_scores}

def evaluate(score_fn) -> pd.DataFrame:
    rows = []
    for text, kind, value in queries:
        rel = relevance(kind, value)
        order = np.argsort(-score_fn(text), kind="stable")
        rows.append({"query": text, "kind": kind,
                     "ndcg@10": ndcg_at_k(rel[order], rel, K),
                     "p@10": precision_at_k(rel[order], K)})
    return pd.DataFrame(rows)

results = {name: evaluate(fn) for name, fn in methods.items()}
summary = pd.concat({n: r.groupby("kind")[["ndcg@10", "p@10"]].mean() for n, r in results.items()})
summary.round(3)

Reading guide: with only metadata words in the text, both methods should be near perfect on **genre** names (exact word match),
and the interesting gap is on **styles** and on queries whose wording differs from the label. A high score here proves little;
it only shows the pipeline works.

## 6. Eyeball test: free-text questions (no ground truth yet)

These content queries have no labels, so we **look** at the top 5 of each method. This is how we notice what the metadata
cannot answer (e.g. "stormy sea"), and it feeds the hand-written query deck.

In [ ]:
def top(score_fn, query: str, n: int = 5) -> None:
    order = np.argsort(-score_fn(query), kind="stable")[:n]
    rows = corpus.iloc[order]
    show_ranking(ROOT, rows.path.tolist(), rows.text.tolist(), f"{query!r}")

for q in ["a stormy sea", "portraits of children", "flowers in a vase"]:
    for name, fn in methods.items():
        print(name)
        top(fn, q)

## Next

- Write the **24 dev queries by hand** (attributes, content, compound, no-answer) and grade the pooled top-5 (plan section 4).
- Then R2: add a short generated description per painting and measure the gain on content queries.